
## Folder 07 — File 02: Gold Product Performance:


This notebook builds **product-level business analytics** using **Silver products, orders** and **order items**.

It calculates:

- Total units sold per product.
- Product revenue.
- Number of orders containing each product.
- Average selling price.
- Product cost and estimated gross profit.
- Product profit margin.
- Product category performance.
- Product revenue rankings.

**Business rule:**

- Cancelled orders are excluded. 
- Product revenue is calculated using order_items.
- line_total, while estimated gross profit uses unit_cost × quantity as the cost basis. 
- This is an estimate, not accounting-grade profit.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

PRODUCTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_products"
ORDERS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
ORDER_ITEMS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items"

GOLD_PRODUCT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.gold_product_performance"
GOLD_CATEGORY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.gold_category_performance"

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

print("Run ID:", RUN_ID)


## 3. Validate source tables

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

SOURCE_TABLES = [
    PRODUCTS_TABLE,
    ORDERS_TABLE,
    ORDER_ITEMS_TABLE
]

for table_name in SOURCE_TABLES:
    if not spark.catalog.tableExists(table_name):
        raise Exception(f"Missing source table: {table_name}")

print("All source tables are available.")
print("Gold schema is ready.")


## 4. Load Silver datasets

products_df = spark.table(PRODUCTS_TABLE)
orders_df = spark.table(ORDERS_TABLE)
order_items_df = spark.table(ORDER_ITEMS_TABLE)

print("Products:", products_df.count())
print("Orders:", orders_df.count())
print("Order items:", order_items_df.count())


## 5. Validate required columns

required_product_columns = [
    "product_id",
    "product_name",
    "category",
    "unit_price",
    "unit_cost"
]

required_order_columns = [
    "order_id",
    "order_status"
]

required_item_columns = [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_total"
]

for column_name in required_product_columns:
    if column_name not in products_df.columns:
        raise Exception(f"Missing product column: {column_name}")

for column_name in required_order_columns:
    if column_name not in orders_df.columns:
        raise Exception(f"Missing order column: {column_name}")

for column_name in required_item_columns:
    if column_name not in order_items_df.columns:
        raise Exception(f"Missing order item column: {column_name}")

print("Required source columns validated.")


## 6. Clean product data

products_clean_df = (
    products_df
    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )
    .withColumn(
        "product_name",
        F.trim(F.col("product_name"))
    )
    .withColumn(
        "category",
        F.coalesce(
            F.initcap(F.trim(F.col("category"))),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(18,2)")
    )
    .withColumn(
        "unit_cost",
        F.col("unit_cost").cast("decimal(18,2)")
    )
    .filter(
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
        & F.col("unit_price").isNotNull()
        & (F.col("unit_price") >= 0)
        & F.col("unit_cost").isNotNull()
        & (F.col("unit_cost") >= 0)
    )
)

print("Clean product records:", products_clean_df.count())


## 7. Deduplicate product records

# Use the latest available timestamp when one exists.
# If no timestamp exists, conflicting product records cause an error, instead of selecting an arbitrary record.


def safe_deduplicate(
    df,
    key_columns,
    comparison_columns,
    timestamp_candidates,
    dataset_name
):

    available_timestamps = [
        column_name
        for column_name in timestamp_candidates
        if column_name in df.columns
    ]

    if available_timestamps:

        timestamp_column = available_timestamps[0]

        tie_break_columns = [
            F.col(column_name).asc_nulls_last()
            for column_name in comparison_columns
        ]

        dedup_window = (
            Window
            .partitionBy(*key_columns)
            .orderBy(
                F.col(timestamp_column).desc_nulls_last(),
                *tie_break_columns
            )
        )

        result_df = (
            df
            .withColumn(
                "_row_number",
                F.row_number().over(dedup_window)
            )
            .filter(F.col("_row_number") == 1)
            .drop("_row_number")
        )

    else:

        conflict_df = (
            df
            .groupBy(*key_columns)
            .agg(
                F.countDistinct(
                    F.struct(
                        *[
                            F.col(column_name)
                            for column_name in comparison_columns
                        ]
                    )
                ).alias("_distinct_records")
            )
            .filter(F.col("_distinct_records") > 1)
        )

        conflict_count = conflict_df.count()

        if conflict_count > 0:
            raise Exception(
                f"{dataset_name} contains {conflict_count} "
                "conflicting duplicate keys and has no usable "
                "timestamp for deterministic deduplication."
            )

        result_df = df.dropDuplicates(key_columns)

    return result_df


products_deduplicated_df = safe_deduplicate(
    df=products_clean_df,
    key_columns=["product_id"],
    comparison_columns=[
        "product_name",
        "category",
        "unit_price",
        "unit_cost"
    ],
    timestamp_candidates=[
        "_ingestion_timestamp",
        "updated_at",
        "updated_timestamp",
        "created_at"
    ],
    dataset_name="Products"
)

print(
    "Deduplicated products:",
    products_deduplicated_df.count()
)


## 8. Prepare and deduplicate order status data

# Prefer the latest available timestamp.
# If duplicate orders have conflicting statuses and no timestamp,
# stop the pipeline rather than selecting an arbitrary status.

orders_clean_df = (
    orders_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )
    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )
    .filter(
        F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("order_status").isNotNull()
    )
)

orders_deduplicated_df = safe_deduplicate(
    df=orders_clean_df,
    key_columns=["order_id"],
    comparison_columns=["order_status"],
    timestamp_candidates=[
        "_ingestion_timestamp",
        "updated_at",
        "updated_timestamp",
        "order_timestamp",
        "created_at"
    ],
    dataset_name="Orders"
)

orders_status_df = orders_deduplicated_df.select(
    "order_id",
    "order_status"
)

print("Deduplicated orders:", orders_status_df.count())


## 9. Clean and deduplicate order items

order_items_clean_df = (
    order_items_df
    .withColumn(
        "order_item_id",
        F.trim(F.col("order_item_id").cast("string"))
    )
    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )
    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(18,2)")
    )
    .withColumn(
        "line_total",
        F.col("line_total").cast("decimal(18,2)")
    )
    .filter(
        F.col("order_item_id").isNotNull()
        & (F.col("order_item_id") != "")
        & F.col("order_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("quantity").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("unit_price").isNotNull()
        & (F.col("unit_price") >= 0)
        & F.col("line_total").isNotNull()
        & (F.col("line_total") >= 0)
    )
)

order_items_deduplicated_df = safe_deduplicate(
    df=order_items_clean_df,
    key_columns=["order_item_id"],
    comparison_columns=[
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "line_total"
    ],
    timestamp_candidates=[
        "_ingestion_timestamp",
        "updated_at",
        "updated_timestamp",
        "created_at"
    ],
    dataset_name="Order items"
)

print(
    "Clean order items:",
    order_items_clean_df.count()
)

print(
    "Deduplicated order items:",
    order_items_deduplicated_df.count()
)


## 10. Join order items with valid orders

# Exclude cancelled orders from product sales analytics.

valid_order_items_df = (
    order_items_deduplicated_df.alias("items")
    .join(
        orders_status_df.alias("orders"),
        F.col("items.order_id") == F.col("orders.order_id"),
        "inner"
    )
    .filter(
        F.col("orders.order_status") != "CANCELLED"
    )
    .select(
        F.col("items.order_item_id"),
        F.col("items.order_id"),
        F.col("items.product_id"),
        F.col("items.quantity"),
        F.col("items.unit_price"),
        F.col("items.line_total")
    )
)

print(
    "Valid non-cancelled order items:",
    valid_order_items_df.count()
)


## 11. Check unmatched product references

# Identify order items whose product ID does not exist in the cleaned product master.


unmatched_product_items_df = (
    valid_order_items_df
    .join(
        products_deduplicated_df.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

unmatched_product_item_count = unmatched_product_items_df.count()

print(
    "Order items with unmatched product IDs:",
    unmatched_product_item_count
)

if unmatched_product_item_count > 0:
    print(
        "WARNING: These order items will not contribute "
        "to product or category performance."
    )


## 12. Aggregate product sales

# Aggregate order items before joining the product master.
# This prevents product master records from multiplying sales.

product_sales_df = (
    valid_order_items_df
    .groupBy("product_id")
    .agg(
        F.sum("quantity").alias("total_units_sold"),

        F.sum("line_total")
        .cast("decimal(18,2)")
        .alias("total_revenue"),

        F.countDistinct("order_id").alias("total_orders"),

        F.countDistinct("order_item_id").alias("total_order_items"),

        F.avg("unit_price")
        .cast("decimal(18,2)")
        .alias("average_item_unit_price")
    )
)

print(
    "Products with sales:",
    product_sales_df.count()
)


## 13. Join product master with sales

# LEFT JOIN retains products with zero sales.

product_performance_df = (
    products_deduplicated_df.alias("products")
    .join(
        product_sales_df.alias("sales"),
        F.col("products.product_id") == F.col("sales.product_id"),
        "left"
    )
    .select(
        F.col("products.product_id"),
        F.col("products.product_name"),
        F.col("products.category"),
        F.col("products.unit_price"),
        F.col("products.unit_cost"),

        F.coalesce(
            F.col("sales.total_units_sold"),
            F.lit(0)
        ).alias("total_units_sold"),

        F.coalesce(
            F.col("sales.total_revenue"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("total_revenue"),

        F.coalesce(
            F.col("sales.total_orders"),
            F.lit(0)
        ).alias("total_orders"),

        F.coalesce(
            F.col("sales.total_order_items"),
            F.lit(0)
        ).alias("total_order_items"),

        F.col("sales.average_item_unit_price")
    )
)


## 14. Calculate estimated product cost and profit

# Assumption: current product unit_cost is used as the cost basis for all historical units sold.
# This is an estimate, not historical inventory valuation.

product_performance_df = (
    product_performance_df
    .withColumn(
        "estimated_total_cost",
        (
            F.col("unit_cost")
            * F.col("total_units_sold")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_gross_profit",
        (
            F.col("total_revenue")
            - F.col("estimated_total_cost")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_profit_margin_percentage",
        F.when(
            F.col("total_revenue") > 0,
            F.round(
                F.col("estimated_gross_profit")
                / F.col("total_revenue") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)


## 15. Calculate average selling price

product_performance_df = (
    product_performance_df
    .withColumn(
        "average_selling_price",
        F.when(
            F.col("total_units_sold") > 0,
            F.round(
                F.col("total_revenue")
                / F.col("total_units_sold"),
                2
            )
        ).otherwise(F.lit(0))
    )
)


## 16. Rank products by revenue within category

# Ranking restarts within each category.
# MAGIC This avoids a global window that moves all products into a single partition.

# COMMAND ----------

category_revenue_window = (
    Window
    .partitionBy("category")
    .orderBy(
        F.col("total_revenue").desc()
    )
)

product_performance_df = (
    product_performance_df
    .withColumn(
        "category_revenue_rank",
        F.dense_rank().over(category_revenue_window)
    )
)


## 17. Add product performance category and audit columns

product_performance_df = (
    product_performance_df
    .withColumn(
        "performance_category",
        F.when(
            F.col("total_units_sold") == 0,
            "NO_SALES"
        )
        .when(
            F.col("total_revenue") > 0,
            "GENERATING_SALES"
        )
        .otherwise("ZERO_REVENUE")
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)


## 18. Create category performance table

# Calculate category revenue, units, costs and profit.

# Calculate unique category orders independently using order IDs.

# Do not sum product-level order counts because one order can contain multiple products from the same category.

category_sales_summary_df = (
    product_performance_df
    .groupBy("category")
    .agg(
        F.countDistinct("product_id").alias("total_products"),

        F.sum("total_units_sold").alias("total_units_sold"),

        F.sum("total_revenue")
        .cast("decimal(18,2)")
        .alias("total_revenue"),

        F.sum("estimated_total_cost")
        .cast("decimal(18,2)")
        .alias("estimated_total_cost"),

        F.sum("estimated_gross_profit")
        .cast("decimal(18,2)")
        .alias("estimated_gross_profit")
    )
)

category_unique_orders_df = (
    valid_order_items_df.alias("items")
    .join(
        products_deduplicated_df.select(
            "product_id",
            "category"
        ).alias("products"),
        on="product_id",
        how="inner"
    )
    .groupBy("category")
    .agg(
        F.countDistinct("order_id").alias("total_unique_orders")
    )
)

category_performance_df = (
    category_sales_summary_df.alias("summary")
    .join(
        category_unique_orders_df.alias("orders"),
        on="category",
        how="left"
    )
    .withColumn(
        "total_unique_orders",
        F.coalesce(
            F.col("total_unique_orders"),
            F.lit(0)
        )
    )
    .withColumn(
        "estimated_profit_margin_percentage",
        F.when(
            F.col("total_revenue") > 0,
            F.round(
                F.col("estimated_gross_profit")
                / F.col("total_revenue") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)


## 19. Display top 10 products

display(
    product_performance_df
    .orderBy(
        F.col("total_revenue").desc(),
        F.col("product_id").asc()
    )
    .limit(10)
)


## 20. Display category performance

display(
    category_performance_df
    .orderBy(
        F.col("total_revenue").desc()
    )
)


## 21. Save Gold product performance

(
    product_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_PRODUCT_TABLE)
)

print(
    "Product performance table created:",
    GOLD_PRODUCT_TABLE
)


# 22. Save Gold category performance

(
    category_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CATEGORY_TABLE)
)

print(
    "Category performance table created:",
    GOLD_CATEGORY_TABLE
)


## 23. Final validation

gold_product_df = spark.table(GOLD_PRODUCT_TABLE)
gold_category_df = spark.table(GOLD_CATEGORY_TABLE)

gold_product_count = gold_product_df.count()
gold_category_count = gold_category_df.count()

expected_product_count = products_deduplicated_df.count()
expected_category_count = category_performance_df.count()

print("=" * 70)
print("GOLD PRODUCT PERFORMANCE VALIDATION")
print("=" * 70)

print("Expected products:", expected_product_count)
print("Gold product records:", gold_product_count)

print("Expected categories:", expected_category_count)
print("Gold category records:", gold_category_count)

print(
    "Unmatched product order items:",
    unmatched_product_item_count
)

if gold_product_count != expected_product_count:
    raise Exception("Gold product record count validation failed.")

if gold_category_count != expected_category_count:
    raise Exception("Gold category record count validation failed.")

if gold_product_df.filter(
    F.col("product_id").isNull()
).count() > 0:
    raise Exception("Null product IDs found in Gold product table.")

if gold_product_df.groupBy("product_id").count().filter(
    F.col("count") > 1
).count() > 0:
    raise Exception("Duplicate product IDs found in Gold product table.")

print("Gold output validation passed.")


## 24. Product performance distribution

display(
    gold_product_df
    .groupBy("performance_category")
    .agg(
        F.count("*").alias("product_count")
    )
    .orderBy("performance_category")
)


## 25. Final business summary

display(
    gold_product_df
    .agg(
        F.countDistinct("product_id").alias("total_products"),
        F.sum("total_units_sold").alias("total_units_sold"),
        F.sum("total_revenue").alias("total_product_revenue"),
        F.sum("estimated_total_cost").alias("estimated_total_cost"),
        F.sum("estimated_gross_profit").alias("estimated_gross_profit"),
        F.sum(
            F.when(
                F.col("performance_category") == "NO_SALES",
                1
            ).otherwise(0)
        ).alias("products_without_sales")
    )
)

# COMMAND ----------

print("=" * 70)
print("GOLD PRODUCT PERFORMANCE COMPLETED")
print("=" * 70)

print("Run ID:", RUN_ID)
print("Product table:", GOLD_PRODUCT_TABLE)
print("Category table:", GOLD_CATEGORY_TABLE)
print("Product records:", gold_product_count)
print("Category records:", gold_category_count)
print("Unmatched product items:", unmatched_product_item_count)



### Top 20 products by revenue

In [0]:
%sql
SELECT
    product_id,
    product_name,
    category,
    total_units_sold,
    total_revenue,
    estimated_gross_profit,
    estimated_profit_margin_percentage,
    category_revenue_rank
FROM workspace.ecommerce_gold.gold_product_performance
ORDER BY total_revenue DESC
LIMIT 20;


### Category performance

In [0]:
%sql
SELECT
    category,
    total_products,
    total_unique_orders,
    total_units_sold,
    total_revenue,
    estimated_gross_profit,
    estimated_profit_margin_percentage
FROM workspace.ecommerce_gold.gold_category_performance
ORDER BY total_revenue DESC;


### Products with no sales

In [0]:
%sql
SELECT
    product_id,
    product_name,
    category,
    unit_price,
    unit_cost
FROM workspace.ecommerce_gold.gold_product_performance
WHERE performance_category = 'NO_SALES'
ORDER BY category, product_name;


### Validate category order counts

In [0]:
%sql
SELECT
    category,
    total_unique_orders
FROM workspace.ecommerce_gold.gold_category_performance
ORDER BY total_unique_orders DESC;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_product_performance;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_product_performance;